In [2]:
print("My  Logistics Project has started!")

My  Logistics Project has started!


In [3]:
from pathlib import Path

print("Current folder:", Path.cwd())

Current folder: c:\Users\SenthamizhSelvan\OneDrive\Desktop\Prakash\YUVA-Data Analyst\notebooks


In [4]:
from pathlib import Path

project_folder = Path.cwd().parent
raw_folder = project_folder / "data" / "raw"

print("Project folder:", project_folder)
print("Raw data folder exists:", raw_folder.exists())
print("CSV files found:", len(list(raw_folder.glob("*.csv"))))

Project folder: c:\Users\SenthamizhSelvan\OneDrive\Desktop\Prakash\YUVA-Data Analyst
Raw data folder exists: True
CSV files found: 9


In [5]:
for file in raw_folder.glob("*.csv"):
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [6]:
import pandas as pd

orders = pd.read_csv(raw_folder / "olist_orders_dataset.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
print("Number of order rows:", len(orders))
print("Number of columns:", len(orders.columns))

Number of order rows: 99441
Number of columns: 8


In [8]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [9]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [11]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [12]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

print(orders[date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [13]:
orders.loc[
    orders["order_delivered_customer_date"].isnull(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [14]:
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

Duplicate order IDs: 0


In [15]:
total_orders = len(orders)

delivered_orders = (orders["order_status"] == "delivered").sum()

delivered_percentage = (delivered_orders / total_orders) * 100

print("Total orders:", total_orders)
print("Delivered orders:", delivered_orders)
print("Delivered percentage:", round(delivered_percentage, 2), "%")

Total orders: 99441
Delivered orders: 96478
Delivered percentage: 97.02 %


In [16]:
delivered = orders[orders["order_status"] == "delivered"].copy()

print("Orders marked delivered:", len(delivered))
print(
    "Missing customer delivery dates:",
    delivered["order_delivered_customer_date"].isnull().sum()
)

Orders marked delivered: 96478
Missing customer delivery dates: 8


In [17]:
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"]
    - delivered["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

delivered["delivery_days"].describe().round(2)

count    96470.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_days, dtype: float64

In [18]:
delivered["delivery_result"] = "Late"

delivered.loc[
    delivered["order_delivered_customer_date"]
    <= delivered["order_estimated_delivery_date"],
    "delivery_result"
] = "On time"

delivered["delivery_result"].value_counts()

delivery_result
On time    88644
Late        7834
Name: count, dtype: int64

In [19]:
delivery_counts = delivered["delivery_result"].value_counts()

total_classified = delivery_counts.sum()
on_time_count = delivery_counts.get("On time", 0)
late_count = delivery_counts.get("Late", 0)

print("Total classified:", total_classified)
print("On-time orders:", on_time_count)
print("Late orders:", late_count)
print(
    "On-time delivery percentage:",
    round((on_time_count / total_classified) * 100, 2),
    "%"
)

Total classified: 96478
On-time orders: 88644
Late orders: 7834
On-time delivery percentage: 91.88 %


In [20]:
delivered["delivery_result"] = "Unknown"

has_delivery_date = delivered["order_delivered_customer_date"].notna()

delivered.loc[
    has_delivery_date
    & (
        delivered["order_delivered_customer_date"]
        <= delivered["order_estimated_delivery_date"]
    ),
    "delivery_result"
] = "On time"

delivered.loc[
    has_delivery_date
    & (
        delivered["order_delivered_customer_date"]
        > delivered["order_estimated_delivery_date"]
    ),
    "delivery_result"
] = "Late"

print(delivered["delivery_result"].value_counts())

delivery_result
On time    88644
Late        7826
Unknown        8
Name: count, dtype: int64


In [21]:
known_delivery = delivered[
    delivered["delivery_result"].isin(["On time", "Late"])
]

on_time_count = (known_delivery["delivery_result"] == "On time").sum()
late_count = (known_delivery["delivery_result"] == "Late").sum()
total_known = len(known_delivery)

print("Orders with known delivery results:", total_known)
print("On-time orders:", on_time_count)
print("Late orders:", late_count)
print(
    "On-time delivery percentage:",
    round((on_time_count / total_known) * 100, 2),
    "%"
)

Orders with known delivery results: 96470
On-time orders: 88644
Late orders: 7826
On-time delivery percentage: 91.89 %


In [22]:
print("WEEK 1 - INITIAL LOGISTICS FINDINGS")
print("-----------------------------------")
print("Total orders:", len(orders))
print("Orders marked delivered:", len(delivered))
print("Orders with known delivery results:", total_known)
print("Average delivery time (days):", round(delivered["delivery_days"].mean(), 2))
print("On-time delivery rate (%):", round((on_time_count / total_known) * 100, 2))
print("Late orders:", late_count)

WEEK 1 - INITIAL LOGISTICS FINDINGS
-----------------------------------
Total orders: 99441
Orders marked delivered: 96478
Orders with known delivery results: 96470
Average delivery time (days): 12.56
On-time delivery rate (%): 91.89
Late orders: 7826


In [23]:
invalid_estimates = (
    orders["order_estimated_delivery_date"]
    < orders["order_purchase_timestamp"]
)

print("Orders with estimated date before purchase:", invalid_estimates.sum())

Orders with estimated date before purchase: 0


In [24]:
print(
    "Missing purchase dates:",
    orders["order_purchase_timestamp"].isnull().sum()
)

Missing purchase dates: 0


In [25]:
print(
    "Missing estimated delivery dates:",
    orders["order_estimated_delivery_date"].isnull().sum()
)

Missing estimated delivery dates: 0


In [26]:
print(
    "First purchase date:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Last purchase date:",
    orders["order_purchase_timestamp"].max()
)

First purchase date: 2016-09-04 21:15:19
Last purchase date: 2018-10-17 17:30:18


In [27]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

orders["purchase_year"].value_counts().sort_index()

purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [28]:
# WEEK 1: FINAL DATA EXPLORATION SUMMARY

print("=" * 50)
print("WEEK 1 - LOGISTICS DATA EXPLORATION")
print("=" * 50)

# 1. Dataset size
print("\n1. DATASET SIZE")
print("Total orders:", len(orders))
print("Total columns:", len(orders.columns))

# 2. Orders by purchase year
print("\n2. ORDERS BY YEAR")
print(orders["purchase_year"].value_counts().sort_index())

# 3. Order status
print("\n3. ORDER STATUS")
print(orders["order_status"].value_counts())

# 4. Missing values
print("\n4. MISSING VALUES")
print(orders.isnull().sum())

# 5. Duplicate order IDs
print("\n5. DUPLICATE ORDER IDs")
print(orders["order_id"].duplicated().sum())

# 6. Delivery time summary
print("\n6. DELIVERY TIME (DAYS)")
print(delivered["delivery_days"].describe().round(2))

# 7. Delivery performance
print("\n7. DELIVERY PERFORMANCE")
print(delivered["delivery_result"].value_counts())

known_delivery = delivered[
    delivered["delivery_result"].isin(["On time", "Late"])
]

on_time_count = (known_delivery["delivery_result"] == "On time").sum()
late_count = (known_delivery["delivery_result"] == "Late").sum()
total_known = len(known_delivery)

print("Orders with known delivery results:", total_known)
print("On-time delivery rate (%):",
      round(on_time_count / total_known * 100, 2))
print("Late delivery rate (%):",
      round(late_count / total_known * 100, 2))

# 8. Purchase date range
print("\n8. DATASET DATE RANGE")
print("First purchase:", orders["order_purchase_timestamp"].min())
print("Last purchase:", orders["order_purchase_timestamp"].max())

print("\nWEEK 1 PRACTICAL SUMMARY COMPLETED")

WEEK 1 - LOGISTICS DATA EXPLORATION

1. DATASET SIZE
Total orders: 99441
Total columns: 9

2. ORDERS BY YEAR
purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

3. ORDER STATUS
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

4. MISSING VALUES
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
purchase_year                       0
dtype: int64

5. DUPLICATE ORDER IDs
0

6. DELIVERY TIME (DAYS)
count    96470.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_d